# Mid-Review Pilot — 3 Serialization Cells (Notebook Edition)
**Cells:** JSON × fixed × 256 (baseline) · TOON × fixed × 256 · YAML × fixed × 256
**Corpus:** HotpotQA — full context sets of 50 sampled questions, deduplicated (~498 docs)
**Generators:** GPT + Claude · **Judge:** pinned Haiku

## Protocol
1. Place this notebook in the **repo root** (beside your 12 modules).
2. Kernel = **Thesis (.venv)**.
3. **Kernel → Restart Kernel and Run All Cells** — this phrase is your reproducibility statement.
4. Every expensive stage checkpoints to `cache/` — a dead kernel resumes free on rerun.
5. Results are written to `results/hotpotqa/<cell>.json` **with the git commit embedded**,
   matching your existing evidence format.

Prerequisites (once, in the venv):
```
pip install openai anthropic tiktoken pyyaml numpy datasets sentence-transformers faiss-cpu jupyter ipykernel
```
`OPENAI_API_KEY` and `ANTHROPIC_API_KEY` must be set.

In [ ]:
# ============================ CELL 1: CONFIG ============================
import os, json, time, random, subprocess
from pathlib import Path
import numpy as np

SEED         = 42
CHUNK_TOKENS = 256
K            = 5
N_QUESTIONS  = 50
FORMATS      = ["json", "toon", "yaml"]          # baseline first
CELL_NAMES   = {"json": "json_fixed_256", "toon": "toon_fixed_256", "yaml": "yaml_fixed_256"}

GPT_MODEL    = "gpt-4o-mini"
CLAUDE_MODEL = "claude-haiku-4-5-20251001"
CTX_MODEL    = "claude-haiku-4-5-20251001"
JUDGE_MODEL  = "claude-haiku-4-5-20251001"
EMBED_MODEL  = "jinaai/jina-embeddings-v2-base-en"
GEN_PARAMS   = dict(temperature=0, top_p=1, max_tokens=512)

for d in ("data", "cache", "results/hotpotqa"):
    Path(d).mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:
        return "NO-GIT"
COMMIT = git_commit()

assert os.environ.get("OPENAI_API_KEY"),    "OPENAI_API_KEY not set"
assert os.environ.get("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not set"
print(f"Config OK — commit {COMMIT} | seed {SEED} | {len(FORMATS)} cells | {N_QUESTIONS} questions")

## Cell 2 — Import your modules
Runs on **your** code. Fallbacks fire only if a module is missing, with a loud warning —
a fallback run is not valid evidence. Your `serialise.py` must expose `to_json`, `to_toon`,
`to_yaml`, `roundtrip_ok`.

In [ ]:
# ============================ CELL 2: MODULES ============================
import warnings
FALLBACKS = []

try:
    from chunking import fixed_chunks
except ImportError:
    FALLBACKS.append("chunking")
    import tiktoken; _e = tiktoken.get_encoding("cl100k_base")
    def fixed_chunks(text, size_tokens=CHUNK_TOKENS):
        t = _e.encode(text)
        return [_e.decode(t[i:i+size_tokens]) for i in range(0, len(t), size_tokens)]

try:
    from serialise import to_json, to_toon, to_yaml, roundtrip_ok
except ImportError:
    FALLBACKS.append("serialise")
    import yaml as _yaml
    def to_json(r): return json.dumps(r, ensure_ascii=False)
    def _esc(v):
        s = str(v)
        return '"' + s.replace('"', '\\"') + '"' if ("," in s or '"' in s or "\n" in s) else s
    def to_toon(r): return "\n".join(f"{k}: {_esc(v)}" for k, v in r.items())
    def to_yaml(r): return _yaml.safe_dump(r, allow_unicode=True, sort_keys=False)
    def roundtrip_ok(r): return True

try:
    from tokens import count_gpt
except ImportError:
    FALLBACKS.append("tokens")
    import tiktoken; _e2 = tiktoken.get_encoding("cl100k_base")
    def count_gpt(text): return len(_e2.encode(text))

SERIALIZERS = {"json": to_json, "toon": to_toon, "yaml": to_yaml}

if FALLBACKS:
    warnings.warn(f"FALLBACKS ACTIVE: {FALLBACKS} — NOT valid evidence. Run from repo root.")
else:
    print("All thesis modules imported — running on YOUR code.")

## Cell 3 — Dataset: 50 questions, full pooled context (checkpointed)
Every paragraph (gold + distractor) from each sampled question's 10-paragraph context set
becomes a corpus document, deduplicated by title — this is how the ~498-document corpus
is built. No cap: the distractors ARE the retrieval difficulty.

In [ ]:
# ============================ CELL 3: DATA ============================
CORPUS_F, QUEST_F = Path("data/corpus.jsonl"), Path("data/questions.jsonl")

if CORPUS_F.exists() and QUEST_F.exists():
    corpus    = [json.loads(l) for l in CORPUS_F.open(encoding="utf-8")]
    questions = [json.loads(l) for l in QUEST_F.open(encoding="utf-8")]
    print(f"Checkpoint: {len(corpus)} docs, {len(questions)} questions.")
else:
    from datasets import load_dataset
    ds  = load_dataset("hotpot_qa", "distractor", split="validation", trust_remote_code=True)
    idx = random.Random(SEED).sample(range(len(ds)), N_QUESTIONS)
    questions, corpus, seen = [], [], set()
    for i in idx:
        ex = ds[i]
        questions.append({"qid": ex["id"], "question": ex["question"], "answer": ex["answer"],
                          "gold_titles": sorted(set(ex["supporting_facts"]["title"]))})
        for title, sents in zip(ex["context"]["title"], ex["context"]["sentences"]):
            if title not in seen:
                seen.add(title)
                corpus.append({"doc_id": f"doc_{len(corpus):04d}", "title": title,
                               "text": " ".join(sents)})
    with CORPUS_F.open("w", encoding="utf-8") as f:
        for d in corpus: f.write(json.dumps(d, ensure_ascii=False) + "\n")
    with QUEST_F.open("w", encoding="utf-8") as f:
        for q in questions: f.write(json.dumps(q, ensure_ascii=False) + "\n")
    print(f"Built {len(corpus)} docs from {len(questions)} questions' pooled contexts.")

In [ ]:
# ============================ CELL 4: CHUNKING ============================
chunks = []
for doc in corpus:
    for j, piece in enumerate(fixed_chunks(doc["text"], CHUNK_TOKENS)):
        chunks.append({"chunk_id": f'{doc["doc_id"]}_c{j:02d}', "doc_id": doc["doc_id"],
                       "title": doc["title"], "text": piece})
raw_chunk_tokens = sum(count_gpt(c["text"]) for c in chunks)
print(f"{len(chunks)} chunks from {len(corpus)} docs | raw chunk tokens: {raw_chunk_tokens:,}")

## Cell 5 — Contextualization (Option A, checkpointed per snippet)
Snippets from **raw chunk + full document**, before serialization — one snippet set serves
all three formats. Cached per call: kernel death costs nothing.

In [ ]:
# ============================ CELL 5: CONTEXTUALIZATION ============================
import anthropic
_ant = anthropic.Anthropic()

SNIP_F, snippets = Path("cache/snippets.jsonl"), {}
if SNIP_F.exists():
    for l in SNIP_F.open(encoding="utf-8"):
        r = json.loads(l); snippets[r["chunk_id"]] = r["snippet"]
    print(f"Checkpoint: {len(snippets)} snippets cached.")

doc_text = {d["doc_id"]: d["text"] for d in corpus}
todo = [c for c in chunks if c["chunk_id"] not in snippets]
print(f"{len(todo)} snippets to generate ...")

PROMPT = ("<document>\n{doc}\n</document>\n"
          "Here is the chunk we want to situate within the whole document:\n"
          "<chunk>\n{chunk}\n</chunk>\n"
          "Give a short succinct context to situate this chunk within the overall document "
          "for the purposes of improving search retrieval of the chunk. "
          "Answer only with the succinct context and nothing else.")

with SNIP_F.open("a", encoding="utf-8") as f:
    for n, c in enumerate(todo, 1):
        m = _ant.messages.create(model=CTX_MODEL, max_tokens=150, temperature=0,
            system=[{"type": "text", "text": "You write retrieval-context snippets.",
                     "cache_control": {"type": "ephemeral"}}],
            messages=[{"role": "user",
                       "content": PROMPT.format(doc=doc_text[c["doc_id"]][:12000], chunk=c["text"])}])
        snippets[c["chunk_id"]] = m.content[0].text.strip()
        f.write(json.dumps({"chunk_id": c["chunk_id"], "snippet": snippets[c["chunk_id"]]},
                           ensure_ascii=False) + "\n"); f.flush()
        if n % 50 == 0: print(f"  {n}/{len(todo)}")

snippet_tokens = sum(count_gpt(s) for s in snippets.values())
ctx_overhead = 100 * snippet_tokens / raw_chunk_tokens
print(f"{len(snippets)} snippets | contextualization overhead = {ctx_overhead:.1f}%")

## Cell 6 — Serialization × 3 (the independent variable)
Snippet written **inside** the record, then the whole record serialized per format —
the format governs the complete embedded unit.

In [ ]:
# ============================ CELL 6: SERIALIZATION ============================
serialized = {f: [] for f in FORMATS}
tok_index  = {f: 0 for f in FORMATS}
rt_fail = 0

for c in chunks:
    record = {"chunk_id": c["chunk_id"], "title": c["title"],
              "context": snippets[c["chunk_id"]], "text": c["text"]}
    if not roundtrip_ok(record): rt_fail += 1
    for fmt in FORMATS:
        s = SERIALIZERS[fmt](record)
        serialized[fmt].append(s)
        tok_index[fmt] += count_gpt(s)

assert rt_fail == 0, f"{rt_fail} round-trip failures — stop and fix serialise.py"
base = tok_index["json"]
for fmt in FORMATS:
    sav = "" if fmt == "json" else f' | saving vs baseline={100*(1-tok_index[fmt]/base):.1f}%'
    print(f'[{fmt}] indexing tokens={tok_index[fmt]:,}{sav}')

## Cell 7 — Embed + FAISS, one index per format (cached)

In [ ]:
# ============================ CELL 7: EMBED + FAISS ============================
import faiss
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL, trust_remote_code=True)

indexes = {}
for fmt in FORMATS:
    ef = Path(f"cache/emb_{fmt}.npy")
    if ef.exists():
        emb = np.load(ef); print(f"[{fmt}] embeddings from checkpoint {emb.shape}")
    else:
        emb = embedder.encode(serialized[fmt], batch_size=16, show_progress_bar=True,
                              convert_to_numpy=True)
        np.save(ef, emb)
    emb = emb.astype("float32"); faiss.normalize_L2(emb)
    ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb)
    indexes[fmt] = ix
    print(f"[{fmt}] FAISS: {ix.ntotal} vectors")

## Cell 8 — Retrieval + per-question IR metrics
Per-question partial recall: |gold titles retrieved| / |gold titles| (matches your R@5=0.50 lines).
MRR from the rank of the first gold title.

In [ ]:
# ============================ CELL 8: RETRIEVAL ============================
q_emb = embedder.encode([q["question"] for q in questions],
                        convert_to_numpy=True).astype("float32")
faiss.normalize_L2(q_emb)

retrieved, ir_per_q = {f: {} for f in FORMATS}, {f: {} for f in FORMATS}
for fmt in FORMATS:
    _, I = indexes[fmt].search(q_emb, K)
    for qi, q in enumerate(questions):
        idxs = I[qi].tolist()
        retrieved[fmt][q["qid"]] = idxs
        titles = [chunks[i]["title"] for i in idxs]
        gold = set(q["gold_titles"])
        r_at5 = len(gold & set(titles)) / len(gold)                       # partial recall
        rr = 0.0
        for rank, t in enumerate(titles):
            if t in gold: rr = 1.0 / (rank + 1); break
        ir_per_q[fmt][q["qid"]] = {"r5": r_at5, "rr": rr}
    print(f'[{fmt}] Recall@{K}={np.mean([v["r5"] for v in ir_per_q[fmt].values()]):.2f}  '
          f'MRR={np.mean([v["rr"] for v in ir_per_q[fmt].values()]):.4f}')

## Cell 9 — Generation: GPT + Claude × 3 formats (checkpointed per answer)
Context is passed in the cell's own format. Pinned: temperature=0, top_p=1, max_tokens=512,
seed=42 on GPT (Anthropic API has no seed parameter — the documented asymmetry).
Per-query latency is recorded here.

In [ ]:
# ============================ CELL 9: GENERATION ============================
from openai import OpenAI
_oai = OpenAI()

GEN_F, answers = Path("cache/answers.jsonl"), {}
if GEN_F.exists():
    for l in GEN_F.open(encoding="utf-8"):
        r = json.loads(l); answers[(r["fmt"], r["gen"], r["qid"])] = r
    print(f"Checkpoint: {len(answers)} answers cached.")

SYS = ("Answer the question using ONLY the provided context records. "
       "If the context is insufficient, say so. Be concise.")
def build_context(fmt, qid):
    return "\n\n".join(serialized[fmt][i] for i in retrieved[fmt][qid])

with GEN_F.open("a", encoding="utf-8") as f:
    for fmt in FORMATS:
        done = 0
        for q in questions:
            ctx = build_context(fmt, q["qid"])
            prompt = f"Context records:\n{ctx}\n\nQuestion: {q['question']}"
            for gen in ("gpt", "claude"):
                key = (fmt, gen, q["qid"])
                if key in answers: continue
                t0 = time.time()
                if gen == "gpt":
                    r = _oai.chat.completions.create(model=GPT_MODEL, seed=SEED, **GEN_PARAMS,
                        messages=[{"role": "system", "content": SYS},
                                  {"role": "user", "content": prompt}])
                    ans = r.choices[0].message.content
                else:
                    r = _ant.messages.create(model=CLAUDE_MODEL, system=SYS, **GEN_PARAMS,
                        messages=[{"role": "user", "content": prompt}])
                    ans = r.content[0].text
                rec = {"fmt": fmt, "gen": gen, "qid": q["qid"], "answer": ans,
                       "tok_ctx": count_gpt(ctx), "latency_s": round(time.time() - t0, 2)}
                answers[key] = rec
                f.write(json.dumps(rec, ensure_ascii=False) + "\n"); f.flush(); done += 1
        print(f"[{fmt}] generation done ({done} new calls)")

## Cell 10 — Faithfulness judge (pinned, checkpointed)

In [ ]:
# ============================ CELL 10: FAITHFULNESS ============================
JUD_F, judgments = Path("cache/judgments.jsonl"), {}
if JUD_F.exists():
    for l in JUD_F.open(encoding="utf-8"):
        r = json.loads(l); judgments[(r["fmt"], r["gen"], r["qid"])] = r["score"]
    print(f"Checkpoint: {len(judgments)} judgments cached.")

JP = ("Context:\n{ctx}\n\nAnswer:\n{ans}\n\n"
      "What fraction of the factual claims in the Answer are directly supported by the "
      "Context? Respond with ONLY a number between 0 and 1.")

with JUD_F.open("a", encoding="utf-8") as f:
    for key, rec in answers.items():
        if key in judgments: continue
        fmt, gen, qid = key
        r = _ant.messages.create(model=JUDGE_MODEL, max_tokens=10, temperature=0,
            messages=[{"role": "user", "content": JP.format(ctx=build_context(fmt, qid),
                                                            ans=rec["answer"])}])
        try:
            score = max(0.0, min(1.0, float(r.content[0].text.strip().split()[0])))
        except ValueError:
            score = float("nan")
        judgments[key] = score
        f.write(json.dumps({"fmt": fmt, "gen": gen, "qid": qid, "score": score}) + "\n")
        f.flush()
print(f"Judging complete: {len(judgments)} scores.")

## Cell 11 — Per-question lines, result block, results JSON with embedded commit
Reproduces your evidence format exactly: per-question trace, the MID-REVIEW result block,
and one `results/hotpotqa/<cell>.json` per cell with `"commit"` inside.

In [ ]:
# ============================ CELL 11: RESULTS ============================
import statistics as stats

# --- per-question trace (baseline cell), matching your screenshot format ---
print("Per-question (json baseline):")
for q in questions:
    m = ir_per_q["json"][q["qid"]]
    a = answers[("json", "gpt", q["qid"])]
    print(f'{q["qid"][:8]}  R@5={m["r5"]:.2f} MRR={m["rr"]:.2f} '
          f'ctx={a["tok_ctx"]}t {a["latency_s"]:.1f}s')

results = {}
for fmt in FORMATS:
    fa = {g: stats.mean([judgments[(fmt, g, q["qid"])] for q in questions
                         if not np.isnan(judgments[(fmt, g, q["qid"])])])
          for g in ("gpt", "claude")}
    cell = {
        "cell": CELL_NAMES[fmt], "commit": COMMIT, "corpus": "hotpotqa",
        "n_docs": len(corpus), "n_questions": len(questions), "n_chunks": len(chunks),
        "recall_at_5": round(float(np.mean([v["r5"] for v in ir_per_q[fmt].values()])), 4),
        "mrr":         round(float(np.mean([v["rr"] for v in ir_per_q[fmt].values()])), 4),
        "faith_gpt":    round(fa["gpt"], 4), "faith_claude": round(fa["claude"], 4),
        "ctx_tokens_per_query": round(stats.mean(
            [answers[(fmt, "gpt", q["qid"])]["tok_ctx"] for q in questions]), 1),
        "indexing_tokens": tok_index[fmt],
        "contextualization_overhead_pct": round(ctx_overhead, 1) if fmt == "json" else None,
        "token_saving_vs_baseline_pct":
            None if fmt == "json" else round(100 * (1 - tok_index[fmt] / tok_index["json"]), 1),
    }
    results[fmt] = cell
    Path(f'results/hotpotqa/{CELL_NAMES[fmt]}.json').write_text(json.dumps(cell, indent=2))

b = results["json"]
print(f"""
=============== MID-REVIEW SECTION - RESULT BLOCK ================
The baseline cell (JSON x fixed-size, 256 tokens) runs end-to-end on a hotpotqa subset ({b["n_docs"]} documents, {b["n_questions"]} questions, {b["n_chunks"]} chunks). Commit {COMMIT}.

Baseline: Recall@5={b["recall_at_5"]:.2f}  MRR={b["mrr"]:.4f}  faith(GPT)={b["faith_gpt"]:.4f}  faith(Claude)={b["faith_claude"]:.4f}
          ctx tokens/query={b["ctx_tokens_per_query"]}  contextualization overhead={b["contextualization_overhead_pct"]}%  indexing tokens={b["indexing_tokens"]}""")
for fmt in FORMATS[1:]:
    c = results[fmt]
    print(f"""
{CELL_NAMES[fmt]}: Recall@5={c["recall_at_5"]:.2f}  MRR={c["mrr"]:.4f}  faith(GPT)={c["faith_gpt"]:.4f}  faith(Claude)={c["faith_claude"]:.4f}
          ctx tokens/query={c["ctx_tokens_per_query"]}  indexing tokens={c["indexing_tokens"]}  token saving vs baseline={c["token_saving_vs_baseline_pct"]}%""")
print("=" * 66)
if COMMIT == "NO-GIT":
    print("WARNING: not a git repo — commit the repo and rerun Cell 11 only.")

## Cell 12 — Commit + tag (PowerShell)
```powershell
git add -A
git commit -m "Pilot: JSON/TOON/YAML x fixed x 256, 50q pooled HotpotQA (notebook, Restart & Run All)"
git tag pilot-midreview-nb
git rev-parse --short HEAD
```
If Cell 11 stamped an old or NO-GIT commit into the JSON files, commit first, then re-run
**Cells 1 and 11 only** (Cell 1 re-reads the hash; Cell 11 rewrites the JSONs — both are
pure reporting, no experiment state is touched).

**Reproducibility line:** results produced by *Restart Kernel & Run All* of
`run_pilot.ipynb` at commit `<hash>` (tag `pilot-midreview-nb`); seeds and generation
parameters pinned in Cell 1.